# 🎤 TESS Emotion Classification using Deep Learning
## RNN, LSTM, GRU - Audio Classification
**Dataset:** Toronto Emotional Speech Set (TESS)

Classifies emotional speech into 7 emotions using MFCC features extracted from audio.

---
## 1. 📦 Import Libraries

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, pickle, glob

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

import librosa
import librosa.display

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, SimpleRNN, LSTM, GRU
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_auc_score, roc_curve)

print(f"TensorFlow: {tf.__version__}")


---
## 2. 📊 Dataset Understanding

TESS contains 2800 WAV files organized in folders by emotion.
Each folder name contains the emotion label (e.g., "OAF_angry", "YAF_happy").

In [ ]:

TESS_PATH = 'TESS Toronto emotional speech set data'

# List all emotion folders
emotion_folders = [d for d in os.listdir(TESS_PATH) if os.path.isdir(os.path.join(TESS_PATH, d))]
print(f"Emotion folders ({len(emotion_folders)}): {emotion_folders}")

# Build file list with labels
file_paths = []
labels = []
for folder in emotion_folders:
    folder_path = os.path.join(TESS_PATH, folder)
    emotion = folder.split('_')[-1].lower()  # Extract emotion from folder name
    for file in os.listdir(folder_path):
        if file.endswith('.wav'):
            file_paths.append(os.path.join(folder_path, file))
            labels.append(emotion)

df = pd.DataFrame({'path': file_paths, 'emotion': labels})
print(f"\nTotal audio files: {len(df)}")
print(f"\nEmotion distribution:\n{df['emotion'].value_counts()}")

# Visualization of class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['emotion'].value_counts().plot(kind='bar', ax=axes[0],
    color=sns.color_palette("husl", df['emotion'].nunique()), edgecolor='black')
axes[0].set_title('Emotion Class Distribution', fontweight='bold')
axes[0].set_ylabel('Count')

# Plot sample waveform
sample_path = df['path'].iloc[0]
y_audio, sr = librosa.load(sample_path, duration=3)
librosa.display.waveshow(y_audio, sr=sr, ax=axes[1], color='#e74c3c')
axes[1].set_title(f'Sample Waveform ({df["emotion"].iloc[0]})', fontweight='bold')
plt.tight_layout()
plt.savefig('tess_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Mel spectrogram of sample
plt.figure(figsize=(10, 4))
mel_spec = librosa.feature.melspectrogram(y=y_audio, sr=sr)
librosa.display.specshow(librosa.power_to_db(mel_spec, ref=np.max),
                         y_axis='mel', x_axis='time', cmap='magma')
plt.colorbar(format='%+2.0f dB')
plt.title('Mel Spectrogram (Sample)', fontweight='bold')
plt.tight_layout()
plt.savefig('tess_spectrogram.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 3. 🔧 Feature Extraction & Preprocessing

**MFCC (Mel Frequency Cepstral Coefficients):**
- Standard feature for speech/audio ML tasks
- Captures the spectral envelope of audio signals
- Mimics human auditory perception (mel scale)
- We extract 40 MFCCs and average them across time for a fixed-length representation

In [ ]:

def extract_mfcc(file_path, n_mfcc=40, max_len=100):
    """Extract MFCC features from audio file.
    
    Args:
        file_path: Path to WAV file
        n_mfcc: Number of MFCC coefficients
        max_len: Max number of time frames (pad/truncate to this)
    
    Returns:
        MFCC array of shape (max_len, n_mfcc) for RNN input
    """
    try:
        audio, sr = librosa.load(file_path, duration=3, sr=22050)
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)
        # Transpose to (time, features) and pad/truncate
        mfcc = mfcc.T
        if mfcc.shape[0] < max_len:
            mfcc = np.pad(mfcc, ((0, max_len - mfcc.shape[0]), (0, 0)), mode='constant')
        else:
            mfcc = mfcc[:max_len]
        return mfcc
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

print("Extracting MFCC features... (this may take a few minutes)")
N_MFCC = 40
MAX_LEN = 100

features = []
valid_labels = []

for i, row in df.iterrows():
    mfcc = extract_mfcc(row['path'], N_MFCC, MAX_LEN)
    if mfcc is not None:
        features.append(mfcc)
        valid_labels.append(row['emotion'])
    if (i + 1) % 500 == 0:
        print(f"  Processed {i + 1}/{len(df)} files...")

X = np.array(features)
y = np.array(valid_labels)
print(f"\nFeature shape: {X.shape}")  # (samples, time_steps, n_mfcc)
print(f"Labels shape: {y.shape}")

# Label encode
le = LabelEncoder()
y_encoded = le.fit_transform(y)
n_classes = len(le.classes_)
print(f"Classes ({n_classes}): {le.classes_}")

# Scale features
original_shape = X.shape
X_flat = X.reshape(-1, X.shape[-1])
scaler = StandardScaler()
X_scaled_flat = scaler.fit_transform(X_flat)
X_scaled = X_scaled_flat.reshape(original_shape)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

y_train_cat = to_categorical(y_train, n_classes)
y_test_cat = to_categorical(y_test, n_classes)

# Flat version for ANN/Dense baseline (optional)
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

print(f"\nX_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"X_train_flat: {X_train_flat.shape}")


---
## 4. 🏗️ Model Building

In [ ]:

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
EPOCHS = 30
BATCH = 32


### Simple RNN

In [ ]:
print("\n" + "=" * 50)
print("MODEL 1: Simple RNN")
print("=" * 50)
model_rnn = Sequential([
    SimpleRNN(128, input_shape=(MAX_LEN, N_MFCC), return_sequences=True),
    Dropout(0.3),
    SimpleRNN(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(n_classes, activation='softmax')
])
model_rnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_rnn.summary()
history_rnn = model_rnn.fit(X_train, y_train_cat, epochs=EPOCHS, batch_size=BATCH,
                            validation_split=0.2, callbacks=[early_stop], verbose=1)


### LSTM

In [ ]:
print("\n" + "=" * 50)
print("MODEL 2: LSTM")
print("=" * 50)
model_lstm = Sequential([
    LSTM(128, input_shape=(MAX_LEN, N_MFCC), return_sequences=True),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(n_classes, activation='softmax')
])
model_lstm.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_lstm.summary()
history_lstm = model_lstm.fit(X_train, y_train_cat, epochs=EPOCHS, batch_size=BATCH,
                              validation_split=0.2, callbacks=[early_stop], verbose=1)


### GRU

In [ ]:
print("\n" + "=" * 50)
print("MODEL 3: GRU")
print("=" * 50)
model_gru = Sequential([
    GRU(128, input_shape=(MAX_LEN, N_MFCC), return_sequences=True),
    Dropout(0.3),
    GRU(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(n_classes, activation='softmax')
])
model_gru.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_gru.summary()
history_gru = model_gru.fit(X_train, y_train_cat, epochs=EPOCHS, batch_size=BATCH,
                            validation_split=0.2, callbacks=[early_stop], verbose=1)


---
## 5. 📈 Training Curves

In [ ]:

models_info = {'RNN': history_rnn, 'LSTM': history_lstm, 'GRU': history_gru}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, (name, hist) in enumerate(models_info.items()):
    axes[0, i].plot(hist.history['accuracy'], label='Train', linewidth=2)
    axes[0, i].plot(hist.history['val_accuracy'], label='Val', linewidth=2)
    axes[0, i].set_title(f'{name} - Accuracy', fontweight='bold')
    axes[0, i].legend()
    axes[1, i].plot(hist.history['loss'], label='Train', linewidth=2)
    axes[1, i].plot(hist.history['val_loss'], label='Val', linewidth=2)
    axes[1, i].set_title(f'{name} - Loss', fontweight='bold')
    axes[1, i].legend()
plt.suptitle('Training vs Validation Curves', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('tess_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 6. 📋 Evaluation

In [ ]:

def evaluate_cls(model, X_te, y_te_cat, name):
    y_pred_prob = model.predict(X_te)
    y_pred = np.argmax(y_pred_prob, axis=1)
    y_true = np.argmax(y_te_cat, axis=1)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    try:
        auc = roc_auc_score(y_te_cat, y_pred_prob, multi_class='ovr', average='weighted')
    except:
        auc = 0.0
    print(f"\n{name}: Acc={acc:.4f}, F1={f1:.4f}, AUC={auc:.4f}")
    print(classification_report(y_true, y_pred, target_names=le.classes_))
    return {'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec,
            'F1-Score': f1, 'ROC-AUC': auc, 'y_pred': y_pred, 'y_pred_prob': y_pred_prob, 'y_true': y_true}

results = [
    evaluate_cls(model_rnn, X_test, y_test_cat, 'RNN'),
    evaluate_cls(model_lstm, X_test, y_test_cat, 'LSTM'),
    evaluate_cls(model_gru, X_test, y_test_cat, 'GRU')
]

# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for i, r in enumerate(results):
    cm = confusion_matrix(r['y_true'], r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=le.classes_, yticklabels=le.classes_)
    axes[i].set_title(f"{r['Model']}", fontweight='bold')
    axes[i].tick_params(axis='x', rotation=45)
plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('tess_confusion.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 7. 📊 Comparative Analysis

In [ ]:

comp = pd.DataFrame([{k: v for k, v in r.items() if k not in ['y_pred', 'y_pred_prob', 'y_true']}
                     for r in results]).set_index('Model')
print("\n" + "=" * 60)
print("MODEL COMPARISON")
print("=" * 60)
print(comp.round(4))

comp[['Accuracy', 'F1-Score', 'ROC-AUC']].plot(kind='bar', figsize=(10, 5), edgecolor='black')
plt.title('Emotion Classification Model Comparison', fontweight='bold')
plt.xticks(rotation=0)
plt.ylim(0, 1.1)
plt.tight_layout()
plt.savefig('tess_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

best = comp['Accuracy'].idxmax()
print(f"\n🏆 Best model: {best} ({comp.loc[best, 'Accuracy']:.4f})")

print("""
COMPARATIVE ANALYSIS - Speech Emotion Recognition:
- MFCC features capture spectral characteristics of speech.
- LSTM/GRU excel at capturing temporal patterns in audio sequences.
- Simple RNN may struggle with longer MFCC sequences (vanishing gradients).
- LSTM's cell state provides a "memory highway" for long-range temporal info.
- GRU achieves similar performance with fewer parameters.
- 7-class emotion classification is inherently challenging due to subtle
  acoustic differences between similar emotions (e.g., sad vs. neutral).
""")


---
## 8. 💾 Model Saving

In [ ]:

os.makedirs('saved_models', exist_ok=True)
model_rnn.save('saved_models/tess_rnn_model.keras')
model_lstm.save('saved_models/tess_lstm_model.keras')
model_gru.save('saved_models/tess_gru_model.keras')

with open('saved_models/tess_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('saved_models/tess_label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
with open('saved_models/tess_config.pkl', 'wb') as f:
    pickle.dump({'n_mfcc': N_MFCC, 'max_len': MAX_LEN}, f)

comp.to_csv('saved_models/tess_comparison_results.csv')
print("✅ All models saved! 🎉 TESS Emotion Classification Notebook Complete!")
